# Este Notebook contiene los scrips que se pueden ejecutar desde el area

# Procesar las ventas diarias

In [ ]:
# Actualizar las ventas diarias
# Importamos las librerias a usar
from clase_reportes_new import ReportClassNew
rc = ReportClassNew()
pipeline =rc.pipeline_bi()

# Informe Mayoristas

Prueba

In [23]:
# Codigo prueba para envio de correos en los informes diarios
# Importamos las librerias a usar
from clase_reportes_new import ReportClassNew
from send_mail import MailSender
import re

rc = ReportClassNew() 
ms = MailSender()

clientes={
    'DANIELA ANGULO':[ {'LUCEGO SAS':'KRIKA'},{'SURTICOSMETICOS HF EU':'SURTICOSMETICOS'},
                                    {'CMX SAS':'CMX'},{'LASKIN S.A':'LASKIN'}, 
                                 ],
            'DIANA RIOS':[{'FARMATODO COLOMBIA SA':'FARMATODO'}, 
                                    {'DISTRIBUIDORA PASTEUR S.A':'PASTEUR'},{'NOVAVENTA S.A.S':'NOVAVENTA'}, 
                                    {'BRECCIA SALUD S.AS.': 'LOCATEL'}, {'PROSALON DISTRIBUCIONES SAS':'PROSALON'} ] 
                    }


#  Generamos todos los reportes una sola vez
dict_reportes = rc.informe_diario_mayoristas(clientes=clientes)
base_vendedores = dict_reportes['Base_Vendedores']
dict_reportes = dict_reportes['Cuerpo_HTML']


# Enviar correos masivos individuales
mayoristas = {
    'ANTIOQUIA': 'wcastro@lapocion.com',
    'OCCIDENTE': 'wcastro@lapocion.com',
    'COSTA Y SANTANDERES': 'wcastro@lapocion.com',
    'BOGOTA SUR': 'wcastro@lapocion.com',
    'BOGOTA NORTE': 'wcastro@lapocion.com',
    'DISTRIBUIDOR': 'wcastro@lapocion.com',
    'DANIELA ANGULO':'wcastro@lapocion.com',
    'DIANA RIOS':'wcastro@lapocion.com'
}
exceptuando = ['DANIELA ANGULO', 'DIANA RIOS']

for zona, correo in mayoristas.items():
    if zona in dict_reportes:
        if zona in exceptuando:
            cc = ['wcastro@lapocion.com']
        else:
            cc = ['wcastro@lapocion.com']

        ms.enviar_correo(
            destinatarios=correo,
            cc=cc,
            asunto=f"Informe Diario de Ventas - Zona {zona.title()}",
            cuerpo_html=dict_reportes[zona]
        )

        print(f"✅ Correo enviado a {zona} | CC: {', '.join(cc)}")

    else:
        print(f"⚠️ No se encontró información para {zona}")



# ===================================================================
# CORREO CONSOLIDADO OPTIMIZADO - SIN PIES DE PÁGINA REPETIDOS
# ===================================================================

def limpiar_html_zona(html_zona):
    """
    Elimina elementos innecesarios del HTML de cada zona:
    - Pie de página de "Reporte automático..."
    - Tags <html>, <head>, <body>
    - Mantiene solo la tabla principal
    """
    # Eliminar el pie de página (varios patrones posibles)
    patrones_pie = [
        r'<tr>\s*<td[^>]*>\s*Este es un reporte automático.*?</td>\s*</tr>',
        r'<td[^>]*>\s*Este es un reporte automático.*?</td>',
        r'<div[^>]*>\s*<p[^>]*>\s*.*?Reporte automático.*?</p>\s*</div>',
        r'<td[^>]*bgcolor=["\']#f4f7f9["\'][^>]*>.*?Reporte automático.*?</td>',
        r'<tr>.*?Reporte automático.*?</tr>',
    ]
    
    html_limpio = html_zona
    for patron in patrones_pie:
        html_limpio = re.sub(patron, '', html_limpio, flags=re.IGNORECASE | re.DOTALL)
    
    # Extraer solo la tabla principal
    try:
        if '<table align="center"' in html_limpio:
            inicio = html_limpio.find('<table align="center"')
            # Buscar el cierre de la tabla principal
            temp = html_limpio[inicio:]
            
            # Contar tags <table> y </table> para encontrar el cierre correcto
            count_open = 0
            count_close = 0
            pos = 0
            
            while pos < len(temp):
                if temp[pos:pos+6] == '<table':
                    count_open += 1
                    pos += 6
                elif temp[pos:pos+8] == '</table>':
                    count_close += 1
                    if count_close == count_open:
                        # Encontramos el cierre correspondiente
                        tabla_limpia = temp[:pos+8]
                        return tabla_limpia
                    pos += 8
                else:
                    pos += 1
            
            # Si no encontramos el cierre exacto, usar heurística
            fin = temp.find('</body>')
            if fin != -1:
                tabla_limpia = temp[:fin].strip()
            else:
                tabla_limpia = temp
        else:
            # Fallback: extraer body
            if "<body" in html_limpio:
                body_zona = html_limpio.split("<body", 1)[1]
                body_zona = body_zona.split(">", 1)[1]
                tabla_limpia = body_zona.split("</body>", 1)[0]
            else:
                tabla_limpia = html_limpio
    
    except Exception as e:
        print(f"⚠️ Error limpiando HTML: {e}")
        tabla_limpia = html_limpio
    
    return tabla_limpia.strip()


html_consolidado = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Informe Diario - Todas las Zonas</title>
</head>
<body style="margin:0; padding:0; background-color:#f4f7f9; font-family:'Segoe UI', Arial, sans-serif;">
    
    <!-- Contenedor Principal Centrado -->
    <table align="center" border="0" cellpadding="0" cellspacing="0" width="100%" style="background-color:#f4f7f9;">
        <tr>
            <td align="center" style="padding:20px 10px;">
                
                <!-- Encabezado Global -->
                <table border="0" cellpadding="0" cellspacing="0" width="650" style="border-collapse:collapse; margin-bottom:30px;">
                    <tr>
                        <td style="text-align:center; padding:25px; background:linear-gradient(135deg, #1a5276 0%, #2471a3 100%); border-radius:8px; box-shadow:0 2px 8px rgba(0,0,0,0.1);">
                            <h1 style="margin:0; color:#ffffff; font-size:24px; letter-spacing:1px; font-weight:600;">
                                📊 INFORME CONSOLIDADO
                            </h1>
                            <p style="margin:8px 0 0 0; color:#ffffff; font-size:14px; opacity:0.9;">
                                Resumen de Todas las Zonas
                            </p>
                        </td>
                    </tr>
                </table>
"""

# Contador de zonas
contador_zonas = 0
max_zonas_por_email = 25  # Aumentado gracias a la optimización

orden_personalizado = [
    "MAYORISTAS",
    "DISTRIBUIDOR",
    "ANTIOQUIA",
    "BOGOTA NORTE",
    "BOGOTA SUR",
    "COSTA Y SANTANDERES",
    "OCCIDENTE",
    "DIANA RIOS",
    "DANIELA ANGULO"
]

dict_reportes_ordenado = {
    k: dict_reportes[k]
    for k in orden_personalizado
    if k in dict_reportes
}


print("\n🔄 Procesando zonas para consolidado...")

for zona, html_zona in  dict_reportes_ordenado.items():
    
    contador_zonas += 1
    if contador_zonas > max_zonas_por_email:
        html_consolidado += """
                <!-- Nota de Límite -->
                <table border="0" cellpadding="0" cellspacing="0" width="650" style="border-collapse:collapse; margin-bottom:30px;">
                    <tr>
                        <td style="padding:20px; background-color:#fff3cd; border-left:4px solid #ffc107; border-radius:4px;">
                            <p style="margin:0; color:#856404; font-size:13px; line-height:1.6;">
                                <strong>Nota:</strong> Para el detalle completo de cada zona, consulta los correos individuales.
                            </p>
                        </td>
                    </tr>
                </table>
        """
        break
    
    # Limpiar HTML de la zona (eliminar pies de página repetidos)
    tabla_limpia = limpiar_html_zona(html_zona)
    
    print(f"  ✓ Zona {contador_zonas}: {zona} - {len(tabla_limpia)} caracteres")
    
    # Agregar zona al consolidado con separador sutil
    html_consolidado += f"""
                <!-- Zona {contador_zonas}: {zona} -->
                {tabla_limpia}
                
                <!-- Separador entre zonas -->
                <table border="0" cellpadding="0" cellspacing="0" width="650" style="border-collapse:collapse; margin:25px 0;">
                    <tr>
                        <td style="height:1px; background:linear-gradient(to right, transparent, #e0e0e0, transparent);"></td>
                    </tr>
                </table>
    """

# Pie de página ÚNICO al final
html_consolidado += """
                
                <!-- Pie de Página Global -->
                <table border="0" cellpadding="0" cellspacing="0" width="650" style="border-collapse:collapse; margin-top:40px;">
                    <tr>
                        <td style="text-align:center; padding:25px; background-color:#f8f9fa; border-radius:8px; border:1px solid #e0e0e0;">
                            <p style="margin:0; font-size:12px; color:#2c3e50; line-height:1.8; font-weight:500;">
                                Reporte automático generado por el Área de Análisis de Datos
                            </p>
                        </td>
                    </tr>
                </table>
                
            </td>
        </tr>
    </table>
    
</body>
</html>
"""

# Calcular tamaño del HTML
import sys
tamano_kb = len(html_consolidado.encode('utf-8')) / 1024
print(f"\n📊 Tamaño del HTML consolidado: {tamano_kb:.2f} KB")

if tamano_kb > 100:
    print(f"⚠️ ADVERTENCIA: El email supera 100KB. Gmail podría truncarlo.")
    print(f"   Considera reducir max_zonas_por_email o enviar en múltiples correos.")
else:
    print(f"✅ Tamaño OK para Gmail (<100KB)")

# Enviar correo consolidado
ms.enviar_correo(
    destinatarios=['wcastro@lapocion.com'],  
    asunto="📊 Informe Diario de Ventas – Todas las Zonas",
    # cc=['andresvasquez@lapocion.com', 'mburgos@lapocion.com'],
    cuerpo_html=html_consolidado
)

print(f"\n✅ Correo consolidado enviado con {min(contador_zonas, max_zonas_por_email)} zonas")


ruta = rc.validar_ruta()
ruta_carp = ruta / 'file'

for i in mayoristas:
    if i in exceptuando:
        continue
    else: 
        base_vendedores[base_vendedores['ZONA'] == i].to_csv(ruta_carp / f'base_vendedores_{i}.csv', index=False, sep=';')
        

Buscando archivos con extensión '.csv' en: G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\CLEAN DATA
  - Archivo 'Ventas_Junio_2024_Diciembre_2024.csv' leído correctamente.
  - Archivo 'Ventas_Enero_2025_Diciembre_2025.csv' leído correctamente.
  - Archivo 'Ventas_Enero_2026_Febrero_2026.csv' leído correctamente.
Concatenando todos los archivos...
¡Consolidación completada!
📨 Enviando correo a: wcastro@lapocion.com
✅ Correo enviado correctamente.
✅ Correo enviado a ANTIOQUIA | CC: wcastro@lapocion.com
📨 Enviando correo a: wcastro@lapocion.com
✅ Correo enviado correctamente.
✅ Correo enviado a OCCIDENTE | CC: wcastro@lapocion.com
📨 Enviando correo a: wcastro@lapocion.com
✅ Correo enviado correctamente.
✅ Correo enviado a COSTA Y SANTANDERES | CC: wcastro@lapocion.com
📨 Enviando correo a: wcastro@lapocion.com
✅ Correo enviado correctamente.
✅ Correo enviado a BOGOTA SUR | CC: wcastro@lapocion.com
📨 Enviando correo a: wcastro@lapocion.com
✅ Correo enviado correctamente.
✅ Correo enviado a

Informe

In [20]:
## Codigo para envio de corres del informe diario
# Importamos las librerias a usar
from clase_reportes_new import ReportClassNew
from send_mail import MailSender
import re

rc = ReportClassNew() 
ms = MailSender()

clientes={
    'DANIELA ANGULO':[ {'LUCEGO SAS':'KRIKA'},{'SURTICOSMETICOS HF EU':'SURTICOSMETICOS'},
                                    {'CMX SAS':'CMX'},{'LASKIN S.A':'LASKIN'}, 
                                    # {'COOPERATIVA NACIONAL DE DROGUISTAS DETALLISTAS':'COOPIDROGAS'}
                                    ],
            'DIANA RIOS':[{'FARMATODO COLOMBIA SA':'FARMATODO'}, 
                                    {'DISTRIBUIDORA PASTEUR S.A':'PASTEUR'},{'NOVAVENTA S.A.S':'NOVAVENTA'}, 
                                    {'BRECCIA SALUD S.AS.': 'LOCATEL'}, {'PROSALON DISTRIBUCIONES SAS':'PROSALON'} ] 
                    }


#  Generamos todos los reportes una sola vez
dict_reportes = rc.informe_diario_mayoristas(clientes=clientes)
base_vendedores = dict_reportes['Base_Vendedores']
dict_reportes = dict_reportes['Cuerpo_HTML']

# Enviar correos masivos individuales

mayoristas = {
    'ANTIOQUIA': 'dgomez@lapocion.com',
    'OCCIDENTE': 'kgodoy@lapocion.com',
    'COSTA Y SANTANDERES': 'valeriaortega@lapocion.com',
    'BOGOTA SUR': 'nanaya@lapocion.com',
    'BOGOTA NORTE': 'nanaya@lapocion.com',
    'DISTRIBUIDOR': 'vtorres@lapocion.com',
    'DANIELA ANGULO':'daniangulo@lapocion.com',
    'DIANA RIOS':'dianarios@lapocion.com'
}
exceptuando = ['DANIELA ANGULO', 'DIANA RIOS']

for zona, correo in mayoristas.items():
    if zona in dict_reportes:
        if zona in exceptuando:
            cc = ['andresvasquez@lapocion.com']
        else:
            cc = ['andresvasquez@lapocion.com', 'mburgos@lapocion.com']

        ms.enviar_correo(
            destinatarios=correo,
            cc=cc,
            asunto=f"Informe Diario de Ventas - Zona {zona.title()}",
            cuerpo_html=dict_reportes[zona]
        )

        print(f"✅ Correo enviado a {zona} | CC: {', '.join(cc)}")

    else:
        print(f"⚠️ No se encontró información para {zona}")

# ===================================================================
# CORREO CONSOLIDADO OPTIMIZADO - SIN PIES DE PÁGINA REPETIDOS
# ===================================================================

def limpiar_html_zona(html_zona):
    """
    Elimina elementos innecesarios del HTML de cada zona:
    - Pie de página de "Reporte automático..."
    - Tags <html>, <head>, <body>
    - Mantiene solo la tabla principal
    """
    # Eliminar el pie de página (varios patrones posibles)
    patrones_pie = [
        r'<tr>\s*<td[^>]*>\s*Este es un reporte automático.*?</td>\s*</tr>',
        r'<td[^>]*>\s*Este es un reporte automático.*?</td>',
        r'<div[^>]*>\s*<p[^>]*>\s*.*?Reporte automático.*?</p>\s*</div>',
        r'<td[^>]*bgcolor=["\']#f4f7f9["\'][^>]*>.*?Reporte automático.*?</td>',
        r'<tr>.*?Reporte automático.*?</tr>',
    ]
    
    html_limpio = html_zona
    for patron in patrones_pie:
        html_limpio = re.sub(patron, '', html_limpio, flags=re.IGNORECASE | re.DOTALL)
    
    # Extraer solo la tabla principal
    try:
        if '<table align="center"' in html_limpio:
            inicio = html_limpio.find('<table align="center"')
            # Buscar el cierre de la tabla principal
            temp = html_limpio[inicio:]
            
            # Contar tags <table> y </table> para encontrar el cierre correcto
            count_open = 0
            count_close = 0
            pos = 0
            
            while pos < len(temp):
                if temp[pos:pos+6] == '<table':
                    count_open += 1
                    pos += 6
                elif temp[pos:pos+8] == '</table>':
                    count_close += 1
                    if count_close == count_open:
                        # Encontramos el cierre correspondiente
                        tabla_limpia = temp[:pos+8]
                        return tabla_limpia
                    pos += 8
                else:
                    pos += 1
            
            # Si no encontramos el cierre exacto, usar heurística
            fin = temp.find('</body>')
            if fin != -1:
                tabla_limpia = temp[:fin].strip()
            else:
                tabla_limpia = temp
        else:
            # Fallback: extraer body
            if "<body" in html_limpio:
                body_zona = html_limpio.split("<body", 1)[1]
                body_zona = body_zona.split(">", 1)[1]
                tabla_limpia = body_zona.split("</body>", 1)[0]
            else:
                tabla_limpia = html_limpio
    
    except Exception as e:
        print(f"⚠️ Error limpiando HTML: {e}")
        tabla_limpia = html_limpio
    
    return tabla_limpia.strip()


html_consolidado = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Informe Diario - Todas las Zonas</title>
</head>
<body style="margin:0; padding:0; background-color:#f4f7f9; font-family:'Segoe UI', Arial, sans-serif;">
    
    <!-- Contenedor Principal Centrado -->
    <table align="center" border="0" cellpadding="0" cellspacing="0" width="100%" style="background-color:#f4f7f9;">
        <tr>
            <td align="center" style="padding:20px 10px;">
                
                <!-- Encabezado Global -->
                <table border="0" cellpadding="0" cellspacing="0" width="650" style="border-collapse:collapse; margin-bottom:30px;">
                    <tr>
                        <td style="text-align:center; padding:25px; background:linear-gradient(135deg, #1a5276 0%, #2471a3 100%); border-radius:8px; box-shadow:0 2px 8px rgba(0,0,0,0.1);">
                            <h1 style="margin:0; color:#ffffff; font-size:24px; letter-spacing:1px; font-weight:600;">
                                📊 INFORME CONSOLIDADO
                            </h1>
                            <p style="margin:8px 0 0 0; color:#ffffff; font-size:14px; opacity:0.9;">
                                Resumen de Todas las Zonas
                            </p>
                        </td>
                    </tr>
                </table>
"""

# Contador de zonas
contador_zonas = 0
max_zonas_por_email = 25  # Aumentado gracias a la optimización

orden_personalizado = [
    "MAYORISTAS",
    "DISTRIBUIDOR",
    "ANTIOQUIA",
    "BOGOTA NORTE",
    "BOGOTA SUR",
    "COSTA Y SANTANDERES",
    "OCCIDENTE",
    "DIANA RIOS",
    "DANIELA ANGULO"
]

dict_reportes_ordenado = {
    k: dict_reportes[k]
    for k in orden_personalizado
    if k in dict_reportes
}


print("\n🔄 Procesando zonas para consolidado...")

for zona, html_zona in dict_reportes_ordenado.items():
    
    contador_zonas += 1
    if contador_zonas > max_zonas_por_email:
        html_consolidado += """
                <!-- Nota de Límite -->
                <table border="0" cellpadding="0" cellspacing="0" width="650" style="border-collapse:collapse; margin-bottom:30px;">
                    <tr>
                        <td style="padding:20px; background-color:#fff3cd; border-left:4px solid #ffc107; border-radius:4px;">
                            <p style="margin:0; color:#856404; font-size:13px; line-height:1.6;">
                                <strong>Nota:</strong> Para el detalle completo de cada zona, consulta los correos individuales.
                            </p>
                        </td>
                    </tr>
                </table>
        """
        break
    
    # Limpiar HTML de la zona (eliminar pies de página repetidos)
    tabla_limpia = limpiar_html_zona(html_zona)
    
    print(f"  ✓ Zona {contador_zonas}: {zona} - {len(tabla_limpia)} caracteres")
    
    # Agregar zona al consolidado con separador sutil
    html_consolidado += f"""
                <!-- Zona {contador_zonas}: {zona} -->
                {tabla_limpia}
                
                <!-- Separador entre zonas -->
                <table border="0" cellpadding="0" cellspacing="0" width="650" style="border-collapse:collapse; margin:25px 0;">
                    <tr>
                        <td style="height:1px; background:linear-gradient(to right, transparent, #e0e0e0, transparent);"></td>
                    </tr>
                </table>
    """

# Pie de página ÚNICO al final
html_consolidado += """
                
                <!-- Pie de Página Global -->
                <table border="0" cellpadding="0" cellspacing="0" width="650" style="border-collapse:collapse; margin-top:40px;">
                    <tr>
                        <td style="text-align:center; padding:25px; background-color:#f8f9fa; border-radius:8px; border:1px solid #e0e0e0;">
                            <p style="margin:0; font-size:12px; color:#2c3e50; line-height:1.8; font-weight:500;">
                                Reporte automático generado por el Área de Análisis de Datos
                            </p>
                        </td>
                    </tr>
                </table>
                
            </td>
        </tr>
    </table>
    
</body>
</html>
"""

# Calcular tamaño del HTML
import sys
tamano_kb = len(html_consolidado.encode('utf-8')) / 1024
print(f"\n📊 Tamaño del HTML consolidado: {tamano_kb:.2f} KB")

if tamano_kb > 100:
    print(f"⚠️ ADVERTENCIA: El email supera 100KB. Gmail podría truncarlo.")
    print(f"   Considera reducir max_zonas_por_email o enviar en múltiples correos.")
else:
    print(f"✅ Tamaño OK para Gmail (<100KB)")

# Enviar correo consolidado
ms.enviar_correo(
    destinatarios=['jcorrea@lapocion.com', 'hectoraristizabal@lapocion.com'],  
    asunto="📊 Informe Diario de Ventas – Todas las Zonas",
    cc=['andresvasquez@lapocion.com', 'mburgos@lapocion.com'],
    cuerpo_html=html_consolidado
)

print(f"\n✅ Correo consolidado enviado con {min(contador_zonas, max_zonas_por_email)} zonas")

# Guardar bases de vendedores por zona (exceptuando las zonas de los comerciales individuales)
ruta = rc.validar_ruta()
ruta_carp = ruta / 'file'

for i in mayoristas:
    if i in exceptuando:
        continue
    else: 
        base_vendedores[base_vendedores['ZONA'] == i].to_csv(ruta_carp / f'base_vendedores_{i}.csv', index=False, sep=';')
        

Buscando archivos con extensión '.csv' en: G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\CLEAN DATA
  - Archivo 'Ventas_Junio_2024_Diciembre_2024.csv' leído correctamente.
  - Archivo 'Ventas_Enero_2025_Diciembre_2025.csv' leído correctamente.
  - Archivo 'Ventas_Enero_2026_Febrero_2026.csv' leído correctamente.
Concatenando todos los archivos...
¡Consolidación completada!
📨 Enviando correo a: dgomez@lapocion.com
✅ Correo enviado correctamente.
✅ Correo enviado a ANTIOQUIA | CC: andresvasquez@lapocion.com, mburgos@lapocion.com
📨 Enviando correo a: kgodoy@lapocion.com
✅ Correo enviado correctamente.
✅ Correo enviado a OCCIDENTE | CC: andresvasquez@lapocion.com, mburgos@lapocion.com
📨 Enviando correo a: valeriaortega@lapocion.com
✅ Correo enviado correctamente.
✅ Correo enviado a COSTA Y SANTANDERES | CC: andresvasquez@lapocion.com, mburgos@lapocion.com
📨 Enviando correo a: nanaya@lapocion.com
✅ Correo enviado correctamente.
✅ Correo enviado a BOGOTA SUR | CC: andresvasquez@lapocion.com

# Informe de cartera

In [2]:
categorias = [ 'MAYORISTA NV', 'FARMACIAS',
       'EXTERIOR', 'Surticosmeticos', 'COOPIDROGAS', 'Catalogo',
       'ESPECIALIZADAS', 'Distribuidor', 'KRIKA', 'HOLE COSMETICS',
      ]

In [ ]:
from clase_reportes_new import ReportClassNew
from send_mail import MailSender
rc = ReportClassNew()
ms = MailSender()
html_dicc, df_cartera = rc.informe_cartera(categorias)
ruta = rc.validar_ruta()
ruta_file = ruta / 'file' / 'cartera_procesada.csv'
df_cartera.drop(columns=['LINK FORMS'], inplace=True)
df_cartera.to_csv(ruta_file, index=False, sep=';', decimal=',')


In [7]:
import pandas as pd
hoy = pd.to_datetime('today')

In [12]:
hoy.day_name(locale='es_ES')

'Jueves'

In [ ]:

# mayoristas = {
#     '__CONSOLIDADO__':'@lapocion.com', 
#     'SHELLSY VELASCO': 'svelasco@lapocion.com',
#     'MIRIAM BURGOS': 'mburgos@lapocion.com',
#     'DANIELA ANGULO':'daniangulo@lapocion.com',
#     'DIANA RIOS':'dianarios@lapocion.com'
# }
mayoristas = {
    '__CONSOLIDADO__':['wcastro@lapocion.com','wcastro@lapocion.com'],
    'SHELLSY VELASCO': 'wcastro@lapocion.com',
    'MIRIAM BURGOS': 'wcastro@lapocion.com',
    'DANIELA DURAN':'wcastro@lapocion.com',
    'DIANA RIOS':'wcastro@lapocion.com'
}

for responsable, correo in mayoristas.items():
    
    # Saltar el consolidado si no es martes
    if responsable == '__CONSOLIDADO__' and hoy.day_name(locale='es_ES') != "Martes":
        continue

    if responsable not in html_dicc:
        print(f"⚠️ No se encontró información para {responsable}")
        continue

    ms.enviar_correo(
        destinatarios=correo,
        # cc=['andresvasquez@lapocion.com'],
        asunto=f"Informe De Cartera - {responsable.title()}",
        cuerpo_html=html_dicc[responsable]
    )

    print(f"✅ Correo enviado a {responsable}")

📨 Enviando correo a: wcastro@lapocion.com, wcastro@lapocion.com
✅ Correo enviado correctamente.
✅ Correo enviado a __CONSOLIDADO__ 
📨 Enviando correo a: wcastro@lapocion.com
✅ Correo enviado correctamente.
✅ Correo enviado a SHELLSY VELASCO 
📨 Enviando correo a: wcastro@lapocion.com
✅ Correo enviado correctamente.
✅ Correo enviado a MIRIAM BURGOS 
📨 Enviando correo a: wcastro@lapocion.com
✅ Correo enviado correctamente.
✅ Correo enviado a DANIELA DURAN 
📨 Enviando correo a: wcastro@lapocion.com
✅ Correo enviado correctamente.
✅ Correo enviado a DIANA RIOS 


# Actualizar Base Contable

In [2]:
# Actualizar la base contable
from send_mail import MailSender
from clase_reportes_new import ReportClassNew
from pathlib import Path
rc = ReportClassNew()
ms = MailSender()
conta, dicc = rc.archivos_contabilidad()

Buscando archivos con extensión '.xlsx' en: G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\contabilidad\odoo
  - Archivo 'Apunte contable (account.move.line) (4).xlsx' leído correctamente.
  - Archivo 'Apunte contable (account.move.line) (3).xlsx' leído correctamente.
  - Archivo 'Apunte contable (account.move.line) (2).xlsx' leído correctamente.
  - Archivo 'Apunte contable (account.move.line) (1).xlsx' leído correctamente.
  - Archivo 'Apunte contable (account.move.line) (6).xlsx' leído correctamente.
  - Archivo 'Apunte contable (account.move.line) (5).xlsx' leído correctamente.
Concatenando todos los archivos...
¡Consolidación completada!
Buscando archivos con extensión '.csv' en: G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\contabilidad\base
  - Archivo 'base_01-01-2025_29-12-2026.csv' leído correctamente.
Concatenando todos los archivos...
¡Consolidación completada!


In [ ]:
# # Enviar correos masivos descomentar el codigo primero
# equipo = {'JOAN SMITH JARAMILLO':'jsjaramillo@lapocion.com', 
#  'ARENAS QUIÑONES LAURA ALEJANDRA':'larenas@lapocion.com', 
#  'Viviana Erazo':'vivianaerazo@lapocion.com', 
#  'Suleima Lugo':'slugo@lapocion.com', 
#  'SANTIAGO CUCHUMBE DIANA MAYERLI': 'dsantiago@lapocion.com', 
#  'HILLARY DIAZ HILLARY ANDREA':'hceballos@lapocion.com' , 
#  'CASTRO JENSI CAMILA': 'jcastro@lapocion.com',
#  'MARQUEZ SALAZAR VALENTINA':'vmarquez@lapocion.com',
#  'DIANA MAYERLY SANTIAGO CUCHUMBRE':'dsantiago@lapocion.com'
# }
# ruta = rc.validar_ruta()
# ruta_contabilidad = ruta / 'data' / 'contabilidad' / 'correcciones'

# cuerpo_html = """
# <html>
# <body style="font-family: Arial, sans-serif; color: #333; line-height: 1.5;">
#     <p>Hola equipo,</p>

#     <p>
#         Espero se encuentren muy bien<br>
#         Les comparto el archivo con los <strong>registros que actualmente no tienen un centro de costo asignado</strong> en Odoo.
#     </p>

#     <p>
#         Les agradezco si pueden revisarlos y actualizar la información cuando tengan un espacio.
#         Cualquier duda o comentario, me pueden escribir sin problema.
#     </p>

#     <p>Gracias por el apoyo</p>

#     <em>Este es un mensaje automático.</em></p>
# </body>
# </html>
# """

# for key, value in dicc.items():
#     if key in equipo.keys():
#         archivo_adjunto = ruta_contabilidad/value
#         archivo_adjunto = archivo_adjunto.as_posix()
#         print( archivo_adjunto)
#         print(equipo[key])
#         ms.enviar_correo(
#         destinatarios=equipo[key],
#         cc=['slugo@lapocion.com'], # Importante copiar a doña Suleima Lugo
#         asunto="Registros sin centro de costo asignado en Odoo",
#         cuerpo_html=cuerpo_html,
#         adjuntos=[archivo_adjunto]

#     )

# Consolidar Nielsen

In [ ]:
# Lee la carpeta de Nielsen y consolida los archivos Excel en un solo DataFrame
import os 
import pandas as pd

carpeta_seleccionada = r'G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\nielseiq'

lista_dataframes = []


for archivo in os.listdir(carpeta_seleccionada):
    if archivo.endswith(f'.{'xlsx'}'):
        ruta_completa = os.path.join(carpeta_seleccionada, archivo)
            
        df_o_dict = pd.read_excel(ruta_completa)
        df_o_dict = df_o_dict[df_o_dict['Unnamed: 2'].notna()]
        new_header = df_o_dict.iloc[0]

        # Paso 2: Crear nuevo DataFrame desde la fila 8 en adelante, usando ese header
        df_o_dict_limpio = df_o_dict[1:]
        df_o_dict_limpio.columns = new_header
        df_o_dict_limpio.reset_index(drop=True, inplace=True)
        df_o_dict_limpio = df_o_dict_limpio[df_o_dict_limpio['Periods'].notna()].reset_index(drop=True)
        df_o_dict_limpio = df_o_dict_limpio.convert_dtypes()

        lista_dataframes.append(df_o_dict_limpio)   

df_concatenado = pd.concat(lista_dataframes, ignore_index=True)



marcas = df_concatenado['MARCAS'].unique()
df_concatenado['MARCA_ORIGEN'] = df_concatenado['MARCAS']
# Función que busca la marca
def obtener_marca(nombre):
    for marca in marcas:
        if str(nombre).upper().startswith(marca):
            return marca
    return 'OTRAS MARCAS'

df_concatenado['MARCAS'] = df_concatenado['ITEM'].apply(obtener_marca)
df_concatenado.loc[df_concatenado['MARCAS']=='TONGOLE', 'MARCAS'] = 'POCION'
df_concatenado['Fecha'] = pd.to_datetime(df_concatenado['Periods'].str.split(' ').str[-1], dayfirst=True)
df_concatenado.to_excel(r'G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\NIELSEN\nielse_consolidado.xlsx', index=False)

Buscando archivos con extensión '.xlsx' en: G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\nielseiq
  - Archivo 'Tratamientos (1).xlsx' leído correctamente.
  - Archivo 'Acondicionadores.xlsx' leído correctamente.
  - Archivo 'Shampoo.xlsx' leído correctamente.
  - Archivo 'Tratamientos.xlsx' leído correctamente.
  - Archivo 'Acondicionadores (1).xlsx' leído correctamente.
  - Archivo 'Shampoo (2).xlsx' leído correctamente.
Concatenando todos los archivos...
¡Consolidación completada!


C:\Users\ANALISTA DATOS\AppData\Local\Temp\ipykernel_20768\3294823027.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  nielse['Fecha'] = pd.to_datetime(nielse['Periods'].str.split(' ').str[-1], dayfirst=True)
